One-vs-one (OvO) maps each class pair ``(a, b)`` to a boolean ``row_mask`` (which samples belong to the pair) and a binary label array over that subset. Because the other classes are dropped, each pair needs its own ``df_parts`` subset and ``CPP`` instance:

In [ ]:
import numpy as np
import aaanalysis as aa

labels = [0, 0, 1, 1, 2, 2]
aa.SequenceFeature.get_labels_ovo(labels)

Full OvO workflow: subset ``df_parts`` per pair, build a ``CPP`` on the subset, run, and collect:

In [ ]:
df_seq = aa.load_dataset(name="DOM_GSEC", n=9)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
multiclass = np.array([i % 3 for i in range(len(df_parts))])

import pandas as pd
frames = []
for (a, b), (row_mask, binary) in aa.SequenceFeature.get_labels_ovo(multiclass).items():
    df_pair = df_parts[row_mask]
    df_feat = aa.CPP(df_parts=df_pair).run(labels=binary, n_filter=3)
    df_feat["pair"] = f"({a},{b})"
    frames.append(df_feat)
pd.concat(frames, ignore_index=True)[["pair", "feature", "abs_auc"]].head()

**What can go wrong?** As for OvR, labels must be integers with more than one class:

In [ ]:
try:
    aa.SequenceFeature.get_labels_ovo([1, 1, 1])
except ValueError as e:
    print("ValueError:", e)